In [1]:
#Day 7, Topic 5: pd.wide_to_long()
import pandas as pd

In [2]:
#Basic Example
# Wide data: sales per quarter
wide = pd.DataFrame({
    'Store': ['A', 'B'],
    'Q1': [100, 200],
    'Q2': [110, 210],
    'Q3': [120, 220],
    'Q4': [130, 230]
})
print("Wide format:")
wide

Wide format:


,Store,Q1,Q2,Q3,Q4
0,A,100,110,120,130
1,B,200,210,220,230


In [3]:
long = pd.wide_to_long(wide, stubnames='Q', i='Store', j='Quarter')
long

,,Q
Store,Quarter,
A,1,100
B,1,200
A,2,110
B,2,210
A,3,120
B,3,220
A,4,130
B,4,230


In [4]:
#Resetting the Index
long_flat = long.reset_index()
long_flat

,Store,Quarter,Q
0,A,1,100
1,B,1,200
2,A,2,110
3,B,2,210
4,A,3,120
5,B,3,220
6,A,4,130
7,B,4,230


In [5]:
#Multiple Stubnames
wide_multi = pd.DataFrame({
    'Store': ['A', 'B'],
    'Q1_Sales': [100, 200],
    'Q2_Sales': [110, 210],
    'Q1_Units': [10, 20],
    'Q2_Units': [11, 21]
})
wide_multi

,Store,Q1_Sales,Q2_Sales,Q1_Units,Q2_Units
0,A,100,110,10,11
1,B,200,210,20,21


In [6]:
wide_multi.columns = ['Store', 'Sales_Q1', 'Sales_Q2', 'Units_Q1', 'Units_Q2']
wide_multi

,Store,Sales_Q1,Sales_Q2,Units_Q1,Units_Q2
0,A,100,110,10,11
1,B,200,210,20,21


In [7]:
long_multi = pd.wide_to_long(wide_multi, stubnames=['Sales', 'Units'], i='Store', j='Quarter', sep='_', suffix='.+')
long_multi

,,Sales,Units
Store,Quarter,,
A,Q1,100,10
B,Q1,200,20
A,Q2,110,11
B,Q2,210,21


In [9]:
#Real‑World Titanic Example

df = pd.read_csv('titanic.csv')

# Create a wide summary: survival counts by class and sex
wide_survival = pd.pivot_table(df, values='PassengerId', index='Pclass', columns=['Sex', 'Survived'],
                               aggfunc='count', fill_value=0)
# Flatten MultiIndex columns
wide_survival.columns = [f"{sex}_{'Surv' if surv else 'Perish'}" for sex, surv in wide_survival.columns]
wide_survival = wide_survival.reset_index()
print("Wide survival data:")
wide_survival

Wide survival data:


,Pclass,female_Perish,female_Surv,male_Perish,male_Surv
0,1,3,91,77,45
1,2,6,70,91,17
2,3,72,72,300,47


In [10]:
long_survival = pd.wide_to_long(wide_survival, stubnames=['female', 'male'], i='Pclass', j='Status', sep='_', suffix='.+').reset_index()
long_survival

,Pclass,Status,female,male
0,1,Perish,3,77
1,2,Perish,6,91
2,3,Perish,72,300
3,1,Surv,91,45
4,2,Surv,70,17
5,3,Surv,72,47


In [11]:
#Practice Activity: pd.wide_to_long()
import pandas as pd

# Simulate wide data: average Age for Pclass, split by Sex and AgeGroup
wide = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'male_Child': [5, 3, 10],
    'male_Adult': [40, 30, 28],
    'female_Child': [4, 2, 8],
    'female_Adult': [38, 28, 24]
})
wide

,Pclass,male_Child,male_Adult,female_Child,female_Adult
0,1,5,40,4,38
1,2,3,30,2,28
2,3,10,28,8,24


In [15]:
"""Use pd.wide_to_long() to reshape this into long format. The stubnames should be 
['male', 'female']. The new column for the suffix (the age group) 
should be named 'AgeGroup'. Use sep='_' and suffix='.+'. Finally, reset the index and display the result."""
# 1. Reshape using wide_to_long
long_df = pd.wide_to_long(
    wide, 
    stubnames=['male', 'female'], # The prefix of your columns
    i='Pclass',                   # The identifier column
    j='AgeGroup',                 # Name for the new column created from the suffix
    sep='_',                      # The character separating prefix and suffix
    suffix='.+',                  # Regex to match non-numeric suffixes (Adult, Child)
)

# 2. Reset the index to flatten the DataFrame
long_df = long_df.reset_index()

# Display the result
long_df

,Pclass,AgeGroup,male,female
0,1,Child,5,4
1,2,Child,3,2
2,3,Child,10,8
3,1,Adult,40,38
4,2,Adult,30,28
5,3,Adult,28,24


In [16]:
# 1. Melt the male and female columns into a single 'Sex' column
melted = long_df.melt(id_vars=['Pclass', 'AgeGroup'], var_name='Sex', value_name='Age')

# 2. Combine Sex and AgeGroup to recreate the original column headers
melted['Col_Name'] = melted['Sex'] + '_' + melted['AgeGroup']

# 3. Pivot back to wide format
wide_recreated = melted.pivot(index='Pclass', columns='Col_Name', values='Age').reset_index()

wide_recreated

Col_Name,Pclass,female_Adult,female_Child,male_Adult,male_Child
0,1,38,4,40,5
1,2,28,2,30,3
2,3,24,8,28,10
